In [ ]:
# load this dataset and show its columns: yt_videos_with_local_transcripts.parquet
import pandas as pd
# Load the dataset
file_path = r"yt_videos_with_local_transcripts.parquet"
df = pd.read_parquet(file_path)
# Show the columns of the dataset
print(df.columns)

In [ ]:
# show the first 5 rows of the dataset in a table view
df.head()

In [ ]:
# create a time series of the number of videos uploaded per month for this list of channels, and plot it
import pandas as pd
import matplotlib.pyplot as plt

channels = [
    "Mr. Marra",
    "Davide Zambelli",
    "ChiaraMaciTv",
    "Canale di Venti",
    "Smibie Channel",
    "The Lady",
    "Debora Fulli",
    "roccotnl",
    "Camihawke",
    "Il Matricomio",
    "Raissa & Momo",
    "Sara La Rusca",
    "THOMAS BASILICO",
    "Contenuti Zero",
    "Valentina Barbieri",
    "AlicelikeAudrey",
    "Christian Manzoni",
]

# Reload the parquet so this cell does not depend on stale notebook state
plot_df = pd.read_parquet(file_path)

# Detect the needed columns
author_col = next((col for col in ["channelTitle", "channel_title", "channelName", "uploader"] if col in plot_df.columns), None)
date_col = next((col for col in ["publishedAt", "upload_date", "published_at"] if col in plot_df.columns), None)
if author_col is None:
    raise KeyError(f"No expected channel column found. Available columns: {list(plot_df.columns)}")
if date_col is None:
    raise KeyError(f"No expected date column found. Available columns: {list(plot_df.columns)}")

# Figure out the dtype and normalize it to timezone-naive datetime
print(f"{date_col} dtype before conversion: {plot_df[date_col].dtype}")
plot_df[date_col] = pd.to_datetime(plot_df[date_col], errors='coerce', utc=True)
plot_df[date_col] = plot_df[date_col].dt.tz_convert(None)
print(f"{date_col} dtype after conversion: {plot_df[date_col].dtype}")

# Keep only the requested channels and build monthly counts
plot_df = plot_df[plot_df[author_col].isin(channels)].copy()
plot_df = plot_df.dropna(subset=[date_col])
plot_df["month"] = plot_df[date_col].dt.to_period("M").dt.to_timestamp("M")
monthly_counts = (
    plot_df.groupby([author_col, "month"]).size().reset_index(name="videos")
)

# Use a small number of multi-line plots instead of one panel per channel
plot_groups = [
    ["Mr. Marra", "Davide Zambelli", "ChiaraMaciTv", "Canale di Venti", "Smibie Channel"],
    ["The Lady", "Debora Fulli", "roccotnl", "Camihawke"],
    ["Il Matricomio", "Raissa & Momo", "Sara La Rusca", "THOMAS BASILICO"],
    ["Contenuti Zero", "Valentina Barbieri", "AlicelikeAudrey", "Christian Manzoni"],
]

for group_index, group in enumerate(plot_groups, start=1):
    fig, ax = plt.subplots(figsize=(14, 6))
    for channel in group:
        channel_series = monthly_counts[monthly_counts[author_col] == channel].sort_values("month")
        ax.plot(channel_series["month"], channel_series["videos"], marker="o", linewidth=1.8, label=channel)

    ax.set_title(f"Monthly video uploads per channel (group {group_index} of {len(plot_groups)})")
    ax.set_xlabel("Month")
    ax.set_ylabel("Videos")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# plot a boxplot showing the distribution of video lengths for videos published after January 1, 2022.
import pandas as pd
import matplotlib.pyplot as plt
# Reload the parquet so this cell does not depend on stale notebook state
boxplot_df = pd.read_parquet(file_path)
boxplot_df = boxplot_df[boxplot_df['publishedAt'] > '2022-01-01']
plt.boxplot(boxplot_df['duration_seconds'], vert=False)
plt.xlabel('Video Length')
plt.ylabel('Distribution')
plt.title('Distribution of Video Lengths (Published After Jan 1, 2022)')
plt.show()


In [ ]:
# do the same but filter videos for the 'is_short' column and plot two graphs one for each value of 'is_short'
import pandas as pd
import matplotlib.pyplot as plt
# Reload the parquet so this cell does not depend on stale notebook state
boxplot_df = pd.read_parquet(file_path)
boxplot_df = boxplot_df[boxplot_df['publishedAt'] > '2022-01-01']
short_videos = boxplot_df[boxplot_df['is_short'] == True]
long_videos = boxplot_df[boxplot_df['is_short'] == False]
# Plot for short videos
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.boxplot(short_videos['duration_seconds'], vert=False)
plt.xlabel('Video Length (seconds)')
plt.xticks(ticks=range(0, int(short_videos['duration_seconds'].max()) + 1, 60))
plt.title('Distribution of Short Video Lengths (Published After Jan 1, 2022)')
# Plot for long videos; the unit of measurement should be minutes, and markers on the horizontal axis should be on 60, 120, 180, etc. (multiples of 60)
plt.subplot(1, 2, 2)
plt.boxplot(long_videos['duration_seconds']/60, vert=False)
plt.xlabel('Video Length (minutes)')
plt.title('Distribution of Long Video Lengths (Published After Jan 1, 2022)')
plt.xticks(ticks=range(0, int(long_videos['duration_seconds'].max()/60) + 1, 60))
plt.tight_layout()
plt.show()
# draw a normal distribution curve for them as well.


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
short_videos['duration_seconds'].plot(kind='hist', bins=40, density=True, alpha=0.6, color='g')
short_videos['duration_seconds'].plot(kind='kde', color='black')
plt.xlabel('Video Length (seconds)')
plt.title('Distribution of Short Video Lengths (Published After Jan 1, 2022)')
plt.subplot(1, 2, 2)
long_videos['duration_seconds']/60.plot(kind='hist', bins=40, density=True, alpha=0.6, color='b')
long_videos['duration_seconds']/60.plot(kind='kde', color='black')
plt.xlabel('Video Length (minutes)')
plt.title('Distribution of Long Video Lengths (Published After Jan 1, 2022)')
plt.tight_layout()
plt.show()

In [ ]:
# show the distribution of content form for the videos in this dataset, using a pie chart. The content form is in the 'is_short' column.
import pandas as pd
import matplotlib.pyplot as plt

content_distribution = boxplot_df['is_short'].value_counts()
plt.pie(content_distribution.values, labels=content_distribution.index, autopct='%1.1f%%')
plt.title('Distribution of Content Forms')
plt.show()